In [ ]:
import os, sys, json, time, subprocess, shutil, platform
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python", platform.python_version(), "| torch", torch.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if DEVICE.type == "cpu":
    print("WARNING: no GPU found. Part B still works but is much slower; switch to a T4 GPU runtime.")

In [ ]:
SEEDS          = [2020, 2021, 2022]
MAX_EPOCHS     = 500
PATIENCE       = 30
RUN_ABLATIONS  = True
WORK = "/kaggle/working"
RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)
print("Results folder:", RESULTS)


In [ ]:
# Official repositories: DPGNN (paper code, for reference) and CRRS (contains the public dating data
# and the authors' adaptation of DPGNN and the baselines to it).
os.chdir(WORK)
for url, folder in [("https://github.com/RUCAIBox/DPGNN.git", "DPGNN"), ("https://github.com/RUCAIBox/CRRS.git", "CRRS")]:
    if not os.path.isdir(folder):
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, folder], check=True)
DATA_PATH = f"{WORK}/CRRS/dataset/libimseti/libimseti.inter"
print(subprocess.run(["git", "-C", "CRRS", "log", "-1", "--format=CRRS commit %h (%cd)"], capture_output=True, text=True).stdout)
print("Data file:", DATA_PATH, f"({os.path.getsize(DATA_PATH) / 1e6:.1f} MB)")

In [ ]:
%%writefile rep_data.py
# ---------------------------------------------------------------------------
# Data preparation for the DPGNN replication on Libimseti.
# Mirrors the protocol of the official CRRS / RecBole-PJF code:
#   * one row per (man, woman) pair; label = 1 means a mutual match
#   * direct = 0: the man acted first, direct = 1: the woman acted first
#   * random-order ("RO") split 80/10/10, grouped by man, RecBole's rounding rule
#   * train / valid / test targets keep only label = 1 rows (mutual matches)
#   * one-sided (label = 0) rows become extra "add" edges in DPGNN's graph,
#     taken from the whole file exactly as the official code does
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd


def load_libimseti(path):
    """Read the CRRS processed file and remap ids to 0..M-1 (men), 0..F-1 (women)."""
    df = pd.read_csv(path, sep="\t")
    df.columns = [c.split(":")[0] for c in df.columns]
    df = df.rename(columns={"male_id": "m_tok", "female_id": "f_tok"})
    m_tokens = np.sort(df["m_tok"].unique())
    f_tokens = np.sort(df["f_tok"].unique())
    df["m"] = np.searchsorted(m_tokens, df["m_tok"]).astype(np.int64)
    df["f"] = np.searchsorted(f_tokens, df["f_tok"]).astype(np.int64)
    df = df[["m", "f", "direct", "biexposure", "label"]].reset_index(drop=True)
    return df, len(m_tokens), len(f_tokens)


def _calcu_split_ids(tot, ratios):
    """RecBole's Dataset._calcu_split_ids: guarantees small groups still get valid/test rows."""
    cnt = [int(ratios[i] * tot) for i in range(len(ratios))]
    cnt[0] = tot - sum(cnt[1:])
    for i in range(1, len(ratios)):
        if cnt[0] <= 1:
            break
        if 0 < ratios[-i] * tot < 1:
            cnt[-i] += 1
            cnt[0] -= 1
    return list(np.cumsum(cnt)[:-1])


def split_by_man(df, ratios=(0.8, 0.1, 0.1), seed=2020):
    """Shuffle all rows, then split each man's rows 80/10/10 (RecBole 'RS' + order 'RO' + group_by user)."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    shuffled = df.iloc[order].reset_index(drop=True)
    part = np.empty(len(shuffled), dtype=np.int8)
    for _, idx in shuffled.groupby("m", sort=False).indices.items():
        cuts = _calcu_split_ids(len(idx), ratios)
        bounds = [0] + cuts + [len(idx)]
        for p in range(3):
            part[idx[bounds[p]:bounds[p + 1]]] = p
    shuffled["part"] = part
    return shuffled


class ReciprocalData:
    """Everything the models and the evaluator need, as plain numpy arrays."""

    def __init__(self, path, seed=2020):
        df, self.n_m, self.n_f = load_libimseti(path)
        self.raw = df
        sp = split_by_man(df, seed=seed)
        self.split = sp
        pos = sp[sp.label == 1]
        self.train_pos = pos[pos.part == 0][["m", "f"]].to_numpy()
        self.valid_pos = pos[pos.part == 1][["m", "f"]].to_numpy()
        self.test_pos = pos[pos.part == 2][["m", "f"]].to_numpy()
        # one-sided interactions (no match). The official code builds these from the full file.
        fail = df[df.label == 0]
        self.add_m = fail[fail.direct == 0][["m", "f"]].to_numpy()  # man liked woman, no match
        self.add_f = fail[fail.direct == 1][["m", "f"]].to_numpy()  # woman liked man, no match
        # training rows of all labels (used by the LightGCN baseline, CRRS sets REMAIN_NEG=True for it)
        self.train_all = sp[sp.part == 0][["m", "f"]].to_numpy()
        # sets for negative sampling: a sampled negative must not be a training match
        self.m2f_train = [set() for _ in range(self.n_m)]
        self.f2m_train = [set() for _ in range(self.n_f)]
        for m, f in self.train_pos:
            self.m2f_train[m].add(f)
            self.f2m_train[f].add(m)

    def history(self, phase, all_train=False):
        """Pairs hidden from ranking: train matches for validation; train + valid matches for test.
        all_train=True also hides one-sided training interactions, as the official code does for models
        trained on them (LightGCN with REMAIN_NEG: True, whose RecBole history holds every training row)."""
        base = self.train_all if all_train else self.train_pos
        return base if phase == "valid" else np.vstack([base, self.valid_pos])

    def targets(self, phase):
        return self.valid_pos if phase == "valid" else self.test_pos

    def summary(self):
        return {
            "men": self.n_m, "women": self.n_f, "interactions": len(self.raw),
            "mutual_matches": int(self.raw.label.sum()),
            "train_matches": len(self.train_pos), "valid_matches": len(self.valid_pos),
            "test_matches": len(self.test_pos),
            "one_sided_by_men": len(self.add_m), "one_sided_by_women": len(self.add_f),
        }


def sample_negatives(data, m_batch, f_batch, rng):
    """Uniform negatives: a woman the man has not matched in training, and a man the woman has not matched."""
    neg_f = rng.integers(0, data.n_f, size=len(m_batch))
    neg_m = rng.integers(0, data.n_m, size=len(f_batch))
    for i in range(len(m_batch)):
        s = data.m2f_train[m_batch[i]]
        while neg_f[i] in s:
            neg_f[i] = rng.integers(0, data.n_f)
    for i in range(len(f_batch)):
        s = data.f2m_train[f_batch[i]]
        while neg_m[i] in s:
            neg_m[i] = rng.integers(0, data.n_m)
    return neg_m, neg_f

In [ ]:
from rep_data import ReciprocalData
data = ReciprocalData(DATA_PATH, seed=2020)
display(pd.DataFrame([data.summary()]).T.rename(columns={0: "count"}))
display(data.raw.head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
kinds = pd.Series({"Mutual match": data.raw.label.sum(),
                   "One-sided, man first": len(data.add_m), "One-sided, woman first": len(data.add_f)})
axes[0].barh(kinds.index[::-1], kinds.values[::-1] / 1e3, color=["#2a78d6", "#eb6834", "#1baf7a"][::-1], height=0.55)
for y, v in enumerate(kinds.values[::-1]):
    axes[0].text(v / 1e3 + 5, y, f"{v:,}", va="center")
axes[0].set_xlabel("Interactions (thousands)"); axes[0].set_title("Interaction types"); axes[0].grid(axis="y", visible=False)
axes[0].set_xlim(0, kinds.max() / 1e3 * 1.3)
deg_m = np.bincount(data.train_pos[:, 0], minlength=data.n_m); deg_f = np.bincount(data.train_pos[:, 1], minlength=data.n_f)
bins = np.arange(0, 61, 2)
axes[1].hist(np.clip(deg_m, 0, 60), bins=bins, color="#2a78d6", alpha=0.7, label="Men")
axes[1].hist(np.clip(deg_f, 0, 60), bins=bins, color="#eb6834", alpha=0.7, label="Women")
axes[1].set_xlabel("Training matches per person (60 = 60 or more)"); axes[1].set_ylabel("People"); axes[1].legend()
axes[1].set_title("Matches per person")
split = pd.Series({"Train": len(data.train_pos), "Validation": len(data.valid_pos), "Test": len(data.test_pos)})
axes[2].bar(split.index, split.values / 1e3, color="#2a78d6", width=0.55)
for x, v in enumerate(split.values):
    axes[2].text(x, v / 1e3 + 0.6, f"{v:,}", ha="center")
axes[2].set_ylabel("Mutual matches (thousands)"); axes[2].set_title("Split of mutual matches (80/10/10 per man)")
axes[2].grid(axis="x", visible=False)
fig.tight_layout(); fig.savefig(f"{RESULTS}/fig_dataset.png", dpi=160); plt.show()

In [ ]:
%%writefile rep_eval.py
# ---------------------------------------------------------------------------
# Evaluation, mirroring RecBole (Recall, Precision, NDCG, MRR @K) and the
# reciprocal metrics added in CRRS's MultiDirectTrainer.evaluate():
#   CRecall    = |pairs found by side A OR side B| / |test match pairs|
#   CPrecision = |pairs found by A OR B| / (K * (users_A + users_B))
#   SRecall    = 2 * |pairs found by A AND B| / |test match pairs|
#   SPrecision = 2 * |pairs found by A AND B| / (K * (users_A + users_B))
#   RNDCG@K    = user-weighted average of the two sides' NDCG@K
# Side A = each man ranks all women; side B = each woman ranks all men.
# Only users with at least one target match are evaluated on a side.
# ---------------------------------------------------------------------------
import numpy as np


def mask_and_topk_numpy(scores, history, k):
    """Reference implementation: hide history pairs, return top-k column indices per row (best first)."""
    s = scores.astype(np.float64, copy=True)
    if len(history):
        s[history[:, 0], history[:, 1]] = -np.inf
    part = np.argpartition(-s, k, axis=1)[:, :k]
    rows = np.arange(s.shape[0])[:, None]
    order = np.argsort(-s[rows, part], axis=1, kind="stable")
    return part[rows, order]


def _side_metrics(topk, targets_by_user, k):
    """topk: [n_rows, k] ranked candidate ids; targets_by_user: dict row -> set of true ids."""
    users = np.array(sorted(targets_by_user.keys()))
    hits = np.zeros((len(users), k), dtype=bool)
    pos_len = np.zeros(len(users), dtype=np.int64)
    for i, u in enumerate(users):
        t = targets_by_user[u]
        pos_len[i] = len(t)
        hits[i] = np.fromiter((c in t for c in topk[u]), dtype=bool, count=k)
    n_hit = hits.sum(1)
    recall = n_hit / pos_len
    precision = n_hit / k
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    dcg = (hits * disc).sum(1)
    idcg = np.cumsum(disc)[np.minimum(pos_len, k) - 1]
    ndcg = dcg / idcg
    first = np.where(hits.any(1), hits.argmax(1) + 1, np.inf)
    mrr = np.where(np.isfinite(first), 1.0 / first, 0.0)
    found = [(u, c) for i, u in enumerate(users) for c, h in zip(topk[u], hits[i]) if h]
    return {"recall": recall.mean(), "precision": precision.mean(), "ndcg": ndcg.mean(),
            "mrr": mrr.mean(), "n_users": len(users)}, found


def evaluate_topk(topk_a, topk_b, targets, k=50):
    """topk_a: [n_men, k] women ids; topk_b: [n_women, k] men ids; targets: [n, 2] (man, woman) pairs."""
    by_m, by_f = {}, {}
    for m, f in targets:
        by_m.setdefault(int(m), set()).add(int(f))
        by_f.setdefault(int(f), set()).add(int(m))
    res_a, found_a = _side_metrics(topk_a, by_m, k)
    res_b, found_b = _side_metrics(topk_b, by_f, k)
    pairs_a = set(found_a)                              # (man, woman)
    pairs_b = {(m, f) for f, m in found_b}              # flip to (man, woman)
    total = len({(int(m), int(f)) for m, f in targets})
    n_users = res_a["n_users"] + res_b["n_users"]
    union, inter = len(pairs_a | pairs_b), len(pairs_a & pairs_b)
    out = {f"A_{key}@{k}" if key != "n_users" else "A_users": v for key, v in res_a.items()}
    out.update({f"B_{key}@{k}" if key != "n_users" else "B_users": v for key, v in res_b.items()})
    out.update({
        f"recall@{k}": (res_a["recall"] + res_b["recall"]) / 2,       # "average of both sides" (CRRS Table)
        f"ndcg@{k}": (res_a["ndcg"] + res_b["ndcg"]) / 2,
        f"precision@{k}": (res_a["precision"] + res_b["precision"]) / 2,
        f"crecall@{k}": union / total,
        f"cprecision@{k}": union / (k * n_users),
        f"srecall@{k}": 2 * inter / total,
        f"sprecision@{k}": 2 * inter / (k * n_users),
        f"rndcg@{k}": (res_a["ndcg"] * res_a["n_users"] + res_b["ndcg"] * res_b["n_users"]) / n_users,
        "true_positive_pairs": union,
        "target_pairs": total,
    })
    return {key: float(v) for key, v in out.items()}


def evaluate_scores_numpy(scores, data, phase, k=50, all_train=False):
    """Full-ranking evaluation from a dense [n_men, n_women] score matrix (reference path, CPU)."""
    hist = data.history(phase, all_train)
    topk_a = mask_and_topk_numpy(scores, hist, k)
    topk_b = mask_and_topk_numpy(scores.T, hist[:, ::-1], k)
    return evaluate_topk(topk_a, topk_b, data.targets(phase), k)

In [ ]:
# Sanity checks of the evaluator before trusting any number it produces
from rep_eval import evaluate_topk, evaluate_scores_numpy, mask_and_topk_numpy
# (1) a tiny hand-worked example: 2 men, 3 women, top 2
r = evaluate_topk(np.array([[0, 1], [2, 0]]), np.array([[0, 1], [0, 1], [1, 0]]), np.array([[0, 1], [1, 0]]), k=2)
assert abs(r["A_ndcg@2"] - 1 / np.log2(3)) < 1e-9 and abs(r["B_mrr@2"] - 0.75) < 1e-9 and r["crecall@2"] == 1.0
# (2) an "oracle" that scores the true test matches highest must reach Recall = CRecall = 1
rng = np.random.default_rng(0)
S = rng.random((data.n_m, data.n_f)).astype(np.float32)
S_oracle = S.copy(); S_oracle[data.test_pos[:, 0], data.test_pos[:, 1]] += 10
r = evaluate_scores_numpy(S_oracle, data, "test")
assert r["recall@50"] > 0.99 and r["crecall@50"] == 1.0
# (3) random scores should score close to zero
r_random = evaluate_scores_numpy(S, data, "test")
print(f"Oracle recall@50 = {r['recall@50']:.3f}; random recall@50 = {r_random['recall@50']:.4f}")
print("Evaluator checks passed.")

In [ ]:
%%writefile rep_models.py
# ---------------------------------------------------------------------------
# Models, written in plain PyTorch (torch.sparse instead of torch_geometric) so they
# run on a fresh Colab runtime with no extra installs.
#   DPGNN    : faithful re-implementation of recbole_pjf/model/dpgnn.py (official code)
#   LightGCN : RecBole LightGCN as configured in CRRS (trains on all interactions)
#   BPRMF    : RecBole BPR as configured in CRRS
# Node layout of DPGNN's graph (same as the official code):
#   [ men-active | women-passive | men-passive | women-active ]
# ---------------------------------------------------------------------------
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def bpr_loss(pos, neg, gamma=1e-10):
    """RecBole BPRLoss."""
    return -torch.log(gamma + torch.sigmoid(pos - neg)).mean()


def emb_loss(*embeddings):
    """RecBole EmbLoss with require_pow=False: sum of L2 norms divided by the batch size."""
    loss = torch.zeros(1, device=embeddings[-1].device)
    for e in embeddings:
        loss = loss + torch.norm(e, p=2)
    return loss / embeddings[-1].shape[0]


def sym_norm_adj(rows, cols, n_nodes, device):
    """D^-1/2 A D^-1/2 from an edge list in which every edge already appears in both directions.
    Degree = number of listed edges per node, exactly as torch_geometric.utils.degree in the official code."""
    rows = torch.as_tensor(rows, dtype=torch.long)
    cols = torch.as_tensor(cols, dtype=torch.long)
    deg = torch.bincount(rows, minlength=n_nodes).float()
    norm = 1.0 / torch.sqrt(torch.where(deg == 0, torch.ones(1), deg))
    w = norm[rows] * norm[cols]
    # A[target, source]: messages flow source -> target and are summed (aggr='add')
    adj = torch.sparse_coo_tensor(torch.stack([cols, rows]), w, (n_nodes, n_nodes)).coalesce()
    return adj.to(device)


def both_ways(a, b):
    return np.concatenate([a, b]), np.concatenate([b, a])


class DPGNN(nn.Module):
    """Dual-Perspective GNN (Yang et al., RecSys 2022) for men (side A) and women (side B)."""

    def __init__(self, data, device, dim=128, n_layers=3, reg_weight=1e-4, mutual_weight=0.05,
                 temperature=0.2, use_add_edges=True, use_mutual_loss=True):
        super().__init__()
        self.n_u, self.n_i, self.device = data.n_m, data.n_f, device
        self.n_layers, self.reg_weight = n_layers, reg_weight
        self.mutual_weight = mutual_weight if use_mutual_loss else 0.0
        self.temperature = temperature
        self.user_a = nn.Embedding(self.n_u, dim)   # man as chooser
        self.item_a = nn.Embedding(self.n_i, dim)   # woman as chooser
        self.user_p = nn.Embedding(self.n_u, dim)   # man as the one chosen
        self.item_p = nn.Embedding(self.n_i, dim)   # woman as the one chosen
        for emb in (self.user_a, self.item_a, self.user_p, self.item_p):
            nn.init.xavier_normal_(emb.weight)
        self.adj = self._build_graph(data, use_add_edges)

    def _build_graph(self, data, use_add_edges):
        n_u, n_i = self.n_u, self.n_i
        n_all = n_u + n_i
        R, C = [], []

        def add(r, c):
            rr, cc = both_ways(r, c)
            R.append(rr); C.append(cc)

        # mutual matches (training only): man-active <-> woman-passive and man-passive <-> woman-active
        m, f = data.train_pos[:, 0], data.train_pos[:, 1] + n_u
        add(m, f)
        add(m + n_all, f + n_all)
        if use_add_edges:
            # one-sided interest by the man: man-active <-> woman-passive
            m, f = data.add_m[:, 0], data.add_m[:, 1] + n_u
            add(m, f)
            # one-sided interest by the woman: woman-active <-> man-passive
            m, f = data.add_f[:, 0], data.add_f[:, 1] + n_u
            add(m + n_all, f + n_all)
        # self edges linking each person's two nodes
        u = np.arange(n_u); i = np.arange(n_i) + n_u
        add(u, u + n_all)
        add(i, i + n_all)
        return sym_norm_adj(np.concatenate(R), np.concatenate(C), 2 * n_all, self.device)

    def ego(self):
        return torch.cat([self.user_a.weight, self.item_p.weight, self.user_p.weight, self.item_a.weight], 0)

    def propagate(self):
        x = self.ego()
        layers = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(self.adj, x)
            layers.append(x)
        out = torch.stack(layers, 1).mean(1)
        return torch.split(out, [self.n_u, self.n_i, self.n_u, self.n_i])  # u_a, i_p, u_p, i_a

    def _info_nce(self, a, p):
        a, p = F.normalize(a, dim=1), F.normalize(p, dim=1)
        sim = a @ p.T
        mask = torch.eye(sim.shape[0], dtype=torch.bool, device=sim.device)
        logits = torch.cat([sim[mask].view(sim.shape[0], -1), sim[~mask].view(sim.shape[0], -1)], 1)
        labels = torch.zeros(sim.shape[0], dtype=torch.long, device=sim.device)
        return F.cross_entropy(logits / self.temperature, labels)

    def loss(self, m, f, neg_m, neg_f):
        u_a, i_p, u_p, i_a = self.propagate()
        r_pos = (u_a[m] * i_p[f]).sum(1)            # man's interest in woman
        s_pos = (u_p[m] * i_a[f]).sum(1)            # woman's interest in man
        r_neg1 = (u_a[m] * i_p[neg_f]).sum(1)
        s_neg1 = (u_p[m] * i_a[neg_f]).sum(1)
        r_neg2 = (u_a[neg_m] * i_p[f]).sum(1)
        s_neg2 = (u_p[neg_m] * i_a[f]).sum(1)
        loss = bpr_loss(2 * r_pos + 2 * s_pos, r_neg1 + s_neg1 + r_neg2 + s_neg2)
        loss = loss + self.reg_weight * emb_loss(
            self.user_a(m), self.user_p(m), self.item_a(f), self.item_p(f),
            self.item_a(neg_f), self.item_p(neg_f), self.user_a(neg_m), self.user_p(neg_m))
        if self.mutual_weight > 0:
            loss = loss + self.mutual_weight * self._info_nce(self.user_a.weight[m], self.user_p.weight[m])
            loss = loss + self.mutual_weight * self._info_nce(self.item_a.weight[f], self.item_p.weight[f])
        return loss

    @torch.no_grad()
    def full_scores(self, mode="both"):
        """[n_men, n_women] scores. mode: 'both' (official), 'man_side' or 'woman_side' (analysis only)."""
        u_a, i_p, u_p, i_a = self.propagate()
        if mode == "man_side":
            return u_a @ i_p.T
        if mode == "woman_side":
            return u_p @ i_a.T
        return u_a @ i_p.T + u_p @ i_a.T


class LightGCN(nn.Module):
    """RecBole LightGCN (He et al., SIGIR 2020) as run in CRRS: one node per person."""

    def __init__(self, data, device, dim=128, n_layers=3, reg_weight=5e-4, train_pairs=None):
        super().__init__()
        self.n_u, self.n_i, self.device = data.n_m, data.n_f, device
        self.n_layers, self.reg_weight = n_layers, reg_weight
        self.user = nn.Embedding(self.n_u, dim)
        self.item = nn.Embedding(self.n_i, dim)
        nn.init.xavier_uniform_(self.user.weight); nn.init.xavier_uniform_(self.item.weight)
        pairs = np.unique(train_pairs, axis=0)
        r, c = both_ways(pairs[:, 0], pairs[:, 1] + self.n_u)
        self.adj = sym_norm_adj(r, c, self.n_u + self.n_i, device)

    def propagate(self):
        x = torch.cat([self.user.weight, self.item.weight], 0)
        layers = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(self.adj, x)
            layers.append(x)
        out = torch.stack(layers, 1).mean(1)
        return torch.split(out, [self.n_u, self.n_i])

    def loss(self, m, f, neg_m, neg_f):
        u, i = self.propagate()
        loss = bpr_loss((u[m] * i[f]).sum(1), (u[m] * i[neg_f]).sum(1))
        return loss + self.reg_weight * emb_loss(self.user(m), self.item(f), self.item(neg_f))

    @torch.no_grad()
    def full_scores(self, mode="both"):
        u, i = self.propagate()
        return u @ i.T


class BPRMF(nn.Module):
    """RecBole BPR (Rendle et al., UAI 2009) as run in CRRS."""

    def __init__(self, data, device, dim=128, **_):
        super().__init__()
        self.user = nn.Embedding(data.n_m, dim)
        self.item = nn.Embedding(data.n_f, dim)
        nn.init.xavier_normal_(self.user.weight); nn.init.xavier_normal_(self.item.weight)

    def loss(self, m, f, neg_m, neg_f):
        u = self.user(m)
        return bpr_loss((u * self.item(f)).sum(1), (u * self.item(neg_f)).sum(1))

    @torch.no_grad()
    def full_scores(self, mode="both"):
        return self.user.weight @ self.item.weight.T

In [ ]:
%%writefile rep_train.py
# ---------------------------------------------------------------------------
# Training loop with the official settings from CRRS (overall.yaml):
#   Adam, lr 1e-3, batch 4096, evaluate every epoch on validation,
#   early stopping on validation CRecall@50 with patience 30, test with the best checkpoint.
# ---------------------------------------------------------------------------
import copy
import random
import time

import numpy as np
import torch

from rep_data import sample_negatives
from rep_eval import evaluate_topk
from rep_models import DPGNN, LightGCN, BPRMF


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def topk_torch(scores, history, k):
    """Same as rep_eval.mask_and_topk_numpy, on the GPU."""
    s = scores.clone()
    if len(history):
        h = torch.as_tensor(history, dtype=torch.long, device=s.device)
        s[h[:, 0], h[:, 1]] = -float("inf")
    return torch.topk(s, k, dim=1).indices.cpu().numpy()


def evaluate_model(model, data, phase, k=50, mode="both", all_train=False):
    model.eval()
    scores = model.full_scores(mode)
    hist = data.history(phase, all_train)
    topk_a = topk_torch(scores, hist, k)
    topk_b = topk_torch(scores.T.contiguous(), hist[:, ::-1].copy(), k)
    model.train()
    return evaluate_topk(topk_a, topk_b, data.targets(phase), k)


MODELS = {"DPGNN": DPGNN, "LightGCN": LightGCN, "BPRMF": BPRMF}


def build_model(name, data, device, **kw):
    if name == "LightGCN":
        kw.setdefault("train_pairs", data.train_all)
    return MODELS[name](data, device, **kw).to(device)


def train(name, data, device, seed=2020, max_epochs=500, patience=30, batch_size=4096, lr=1e-3,
          k=50, valid_metric="crecall@50", verbose=True, **model_kw):
    """Train one model; returns (best model, history list, test metrics, info dict)."""
    set_seed(seed)
    rng = np.random.default_rng(seed)
    model = build_model(name, data, device, **model_kw)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    # LightGCN in CRRS trains on every training interaction (REMAIN_NEG: True); the others on matches only
    pairs = data.train_all if name == "LightGCN" else data.train_pos
    if name == "LightGCN":
        neg_sets_m = [set() for _ in range(data.n_m)]; neg_sets_f = [set() for _ in range(data.n_f)]
        for m, f in pairs:
            neg_sets_m[m].add(f); neg_sets_f[f].add(m)
        sampler_data = type("S", (), {"n_m": data.n_m, "n_f": data.n_f, "m2f_train": neg_sets_m, "f2m_train": neg_sets_f})
    else:
        sampler_data = data
    # LightGCN is trained on every training interaction, so (as in the official code) all of them are hidden when ranking
    all_train = name == "LightGCN"
    best, best_state, best_epoch, wait, history = -1.0, None, 0, 0, []
    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        perm = rng.permutation(len(pairs))
        total, n_batches = 0.0, 0
        for start in range(0, len(perm), batch_size):
            b = pairs[perm[start:start + batch_size]]
            neg_m, neg_f = sample_negatives(sampler_data, b[:, 0], b[:, 1], rng)
            t = lambda x: torch.as_tensor(x, dtype=torch.long, device=device)
            loss = model.loss(t(b[:, 0]), t(b[:, 1]), t(neg_m), t(neg_f))
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); n_batches += 1
        res = evaluate_model(model, data, "valid", k, all_train=all_train)
        score = res[valid_metric]
        history.append({"epoch": epoch, "loss": total / n_batches, **{f"valid_{a}": v for a, v in res.items()}})
        if score > best:
            best, best_epoch, wait = score, epoch, 0
            best_state = copy.deepcopy({a: v.detach().cpu() for a, v in model.state_dict().items()})
        else:
            wait += 1
        if verbose and (epoch % 10 == 0 or epoch == 1 or wait == 0):
            print(f"[{name} seed {seed}] epoch {epoch:4d}  loss {total / n_batches:.4f}  "
                  f"valid recall@50 {res['recall@50']:.4f}  crecall@50 {score:.4f}"
                  f"{'  *best*' if wait == 0 else ''}  ({time.time() - t0:.0f}s)")
        if wait >= patience:
            break
    model.load_state_dict({a: v.to(device) for a, v in best_state.items()})
    test = evaluate_model(model, data, "test", k, all_train=all_train)
    info = {"model": name, "seed": seed, "best_epoch": best_epoch, "epochs_run": epoch,
            "best_valid_" + valid_metric: best, "train_seconds": time.time() - t0, **model_kw}
    info.pop("train_pairs", None)
    if verbose:
        print(f"[{name} seed {seed}] best epoch {best_epoch}; TEST recall@50 {test['recall@50']:.4f}  "
              f"ndcg@50 {test['ndcg@50']:.4f}  crecall@50 {test['crecall@50']:.4f}")
    return model, history, test, info

In [ ]:
%%writefile rep_report.py
# ---------------------------------------------------------------------------
# Results tables and figures for the replication.
# ---------------------------------------------------------------------------
import json
import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Published Libimseti (dating) results, CRRS paper (Yang et al., KDD 2024), average of both sides, top 50.
PUBLISHED = {
    "BPRMF":    {"recall@50": 0.1552, "ndcg@50": 0.0661, "crecall@50": 0.2795},
    "LightGCN": {"recall@50": 0.1510, "ndcg@50": 0.0634, "crecall@50": 0.2549},
    "LFRR":     {"recall@50": 0.1729, "ndcg@50": 0.0706, "crecall@50": 0.3045},
    "DPGNN":    {"recall@50": 0.1825, "ndcg@50": 0.0739, "crecall@50": 0.3007},
}
METRICS = ["recall@50", "ndcg@50", "crecall@50"]
LABELS = {"recall@50": "Recall@50", "ndcg@50": "NDCG@50", "crecall@50": "CRecall@50",
          "srecall@50": "SRecall@50", "rndcg@50": "RNDCG@50"}
# names used by the official code for the same models
OFFICIAL_NAME = {"BPR": "BPRMF", "LightGCN": "LightGCN", "LFRR": "LFRR", "DPGNN": "DPGNN"}

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
C_PUB, C_OFF, C_OURS = "#a3a29c", "#2a78d6", "#eb6834"   # published = neutral reference
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]


def style():
    plt.rcParams.update({
        "font.size": 11, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
        "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
        "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True, "figure.facecolor": "white",
        "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left", "legend.frameon": False,
    })


def official_to_avg(rec):
    """Official JSON (side_A / side_B dicts) -> averaged metrics with our key names."""
    a, b = rec["side_A"], rec["side_B"]
    out = {}
    for key in ("recall@50", "ndcg@50", "precision@50", "mrr@50"):
        if key in a and key in b:
            out[key] = (a[key] + b[key]) / 2
            out["A_" + key], out["B_" + key] = a[key], b[key]
    for src, dst in (("crecall", "crecall@50"), ("srecall", "srecall@50"), ("rndcg@50", "rndcg@50"),
                     ("cprecision", "cprecision@50"), ("sprecision", "sprecision@50")):
        if src in a:
            out[dst] = a[src]
    return out


def ours_summary(runs):
    """runs: list of {'label', 'test': {...}} -> DataFrame of mean and std per label."""
    rows = []
    for r in runs:
        rows.append({"label": r["label"], **{k: v for k, v in r["test"].items() if isinstance(v, float)}})
    df = pd.DataFrame(rows)
    g = df.groupby("label", sort=False)
    return g.mean(numeric_only=True), g.std(numeric_only=True).fillna(0.0), g.size()


def comparison_table(ours_mean, ours_std, official):
    """One row per model: published, official-code run, our re-implementation (mean ± std)."""
    rows = []
    for model in ["DPGNN", "LFRR", "LightGCN", "BPRMF", "Popularity"]:
        row = {"Model": model}
        for m in METRICS:
            pub = PUBLISHED.get(model, {}).get(m)
            off = official.get(model, {}).get(m)
            row[f"{LABELS[m]} published"] = f"{pub:.4f}" if pub is not None else "–"
            row[f"{LABELS[m]} official code"] = f"{off:.4f}" if off is not None else "–"
            if model in ours_mean.index:
                mu, sd = ours_mean.loc[model, m], ours_std.loc[model, m]
                row[f"{LABELS[m]} ours"] = f"{mu:.4f} ± {sd:.4f}" if sd > 0 else f"{mu:.4f}"
            else:
                row[f"{LABELS[m]} ours"] = "–"
        rows.append(row)
    return pd.DataFrame(rows).set_index("Model")


def fig_comparison(ours_mean, ours_std, official, path):
    """Grouped bars: published vs official code vs ours, for the models all three cover."""
    style()
    models = [m for m in ["BPRMF", "LightGCN", "LFRR", "DPGNN"] if m in PUBLISHED]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    w = 0.26
    x = np.arange(len(models))
    for ax, m in zip(axes, METRICS):
        pub = [PUBLISHED[k][m] for k in models]
        off = [official.get(k, {}).get(m, np.nan) for k in models]
        mu = [ours_mean.loc[k, m] if k in ours_mean.index else np.nan for k in models]
        sd = [ours_std.loc[k, m] if k in ours_std.index else 0 for k in models]
        ax.bar(x - w, pub, w - 0.02, color=C_PUB, label="Published (CRRS paper)")
        ax.bar(x, off, w - 0.02, color=C_OFF, label="Official code, our run")
        ax.bar(x + w, mu, w - 0.02, yerr=sd, color=C_OURS, label="Our re-implementation",
               error_kw={"ecolor": INK, "elinewidth": 1, "capsize": 3})
        ax.set_xticks(x, models); ax.set_title(LABELS[m]); ax.grid(axis="x", visible=False)
    fig.suptitle("Libimseti test set: published vs reproduced results (higher is better)",
                 x=0.01, y=0.99, ha="left", fontweight="bold")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(0.01, 0.95), ncol=3, fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.9)); fig.savefig(path, dpi=160); plt.close(fig)


def fig_curves(histories, path, metric="valid_crecall@50"):
    """Validation CRecall@50 and training loss per epoch, one line per model (first seed)."""
    style()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    for (label, h), c in zip(histories.items(), SERIES):
        ep = [r["epoch"] for r in h]
        a1.plot(ep, [r[metric] for r in h], color=c, lw=2, label=label)
        best = int(np.argmax([r[metric] for r in h]))
        a1.plot(ep[best], h[best][metric], "o", color=c, ms=8, mec="white", mew=1.5)
        a2.plot(ep, [r["loss"] for r in h], color=c, lw=2, label=label)
    a1.set_xlabel("Epoch"); a1.set_ylabel("Validation CRecall@50"); a1.legend()
    a1.set_title("Validation CRecall@50 (dot = best epoch, kept for testing)")
    a2.set_xlabel("Epoch"); a2.set_ylabel("Training loss"); a2.set_yscale("log"); a2.legend()
    a2.set_title("Training loss (log scale)")
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def fig_bars(values, errors, title, ylabel, path, highlight=None):
    """Horizontal bars for one metric across variants (ablation / analysis)."""
    style()
    labels = list(values.keys())
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(labels) + 1.4))
    y = np.arange(len(labels))[::-1]
    colors = [C_OURS if l == highlight else C_OFF for l in labels]
    ax.barh(y, list(values.values()), xerr=[errors.get(l, 0) for l in labels], color=colors, height=0.6,
            error_kw={"ecolor": INK, "elinewidth": 1, "capsize": 3})
    for yi, l in zip(y, labels):
        ax.text(values[l] + errors.get(l, 0) + max(values.values()) * 0.01, yi, f"{values[l]:.4f}",
                va="center", color=INK, fontsize=10)
    ax.set_yticks(y, labels); ax.set_xlabel(ylabel); ax.grid(axis="y", visible=False)
    ax.set_xlim(0, max(values.values()) * 1.18); ax.set_title(title)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def fig_sides(ours_mean, path):
    """Side A (men ranking women) vs side B (women ranking men) Recall@50."""
    style()
    models = list(ours_mean.index)
    x = np.arange(len(models)); w = 0.38
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - w / 2, ours_mean["A_recall@50"], w - 0.03, color=SERIES[0], label="Side A: men ranking women")
    ax.bar(x + w / 2, ours_mean["B_recall@50"], w - 0.03, color=SERIES[1], label="Side B: women ranking men")
    ax.set_xticks(x, models, rotation=15, ha="right"); ax.set_ylabel("Recall@50")
    ax.legend(ncol=2, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=10)
    ax.grid(axis="x", visible=False); ax.set_title("Recall@50 for each side of the match (our re-implementation)", pad=30)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=1, default=float)

In [ ]:
from rep_train import train, evaluate_model, topk_torch, set_seed
from rep_report import save_json
# GPU top-k must give exactly the same ranking as the tested numpy reference
S_t = torch.as_tensor(S, device=DEVICE)
hist = data.history("valid")
a, b = topk_torch(S_t, hist, 50), mask_and_topk_numpy(S, hist, 50)
assert np.array_equal(np.take_along_axis(S, a, 1), np.take_along_axis(S, b, 1))
# 2-epoch smoke test of every model: losses must be finite and evaluation must run
for name in ["DPGNN", "LightGCN", "BPRMF"]:
    _, h, test, _ = train(name, data, DEVICE, seed=0, max_epochs=2, patience=5, verbose=False)
    assert all(np.isfinite(x["loss"]) for x in h), name
    print(f"{name:9s} smoke test OK  (2 epochs, test recall@50 {test['recall@50']:.4f})")
del S_t

In [ ]:
RUNS_FILE, HIST_FILE = f"{RESULTS}/ours_runs.json", f"{RESULTS}/ours_histories.json"
RUNS = json.load(open(RUNS_FILE))["runs"] if os.path.exists(RUNS_FILE) else []
HIST = json.load(open(HIST_FILE)) if os.path.exists(HIST_FILE) else {}
MODELS_IN_MEMORY = {}

def done(label, seed):
    return any(r["label"] == label and r["seed"] == seed for r in RUNS)

def run(label, model_name, seed, **kw):
    if done(label, seed):
        print(f"skip {label} seed {seed} (already in {RUNS_FILE})"); return
    model, hist, test, info = train(model_name, data, DEVICE, seed=seed, max_epochs=MAX_EPOCHS, patience=PATIENCE, **kw)
    RUNS.append({"label": label, "model": model_name, "seed": seed, "test": test, "info": info})
    HIST.setdefault(label, hist)
    MODELS_IN_MEMORY[(label, seed)] = model
    save_json({"runs": RUNS}, RUNS_FILE); save_json(HIST, HIST_FILE)

# popularity baseline (no training): score = number of training matches of the man + of the woman
if not done("Popularity", 0):
    from rep_eval import evaluate_scores_numpy
    dm = np.bincount(data.train_pos[:, 0], minlength=data.n_m).astype(np.float32)
    df_ = np.bincount(data.train_pos[:, 1], minlength=data.n_f).astype(np.float32)
    RUNS.append({"label": "Popularity", "model": "Popularity", "seed": 0,
                 "test": evaluate_scores_numpy(dm[:, None] + df_[None, :], data, "test"), "info": {}})
    save_json({"runs": RUNS}, RUNS_FILE)

for seed in SEEDS:
    for name in ["DPGNN", "LightGCN", "BPRMF"]:
        run(name, name, seed)

In [ ]:
ABLATIONS = {
    "DPGNN w/o contrastive loss": dict(use_mutual_loss=False),
    "DPGNN w/o one-sided edges": dict(use_add_edges=False),
    "DPGNN w/o both": dict(use_mutual_loss=False, use_add_edges=False),
}
if RUN_ABLATIONS:
    for seed in SEEDS:
        for label, kw in ABLATIONS.items():
            run(label, "DPGNN", seed, **kw)

In [ ]:
TWOWAY_FILE = f"{RESULTS}/ours_twoway.json"
TWOWAY = json.load(open(TWOWAY_FILE)) if os.path.exists(TWOWAY_FILE) else {}
for seed in SEEDS:
    if str(seed) in TWOWAY:
        continue
    model = MODELS_IN_MEMORY.get(("DPGNN", seed))
    if model is None:   # after a restart: retrain this seed (same seed -> same model)
        model, _, _, _ = train("DPGNN", data, DEVICE, seed=seed, max_epochs=MAX_EPOCHS, patience=PATIENCE, verbose=False)
    TWOWAY[str(seed)] = {mode: evaluate_model(model, data, "test", mode=mode) for mode in ["both", "man_side", "woman_side"]}
    save_json(TWOWAY, TWOWAY_FILE)
print("done")

In [ ]:
from rep_report import (official_to_avg, ours_summary, comparison_table, fig_comparison, fig_curves,
                        fig_bars, fig_sides, OFFICIAL_NAME, LABELS, save_json)
OFFICIAL = {}
for model, our_name in OFFICIAL_NAME.items():
    f = f"{RESULTS}/official_{model}.json"
    if os.path.exists(f):
        OFFICIAL[our_name] = official_to_avg(json.load(open(f)))
mean, std, n = ours_summary(RUNS)
table = comparison_table(mean, std, OFFICIAL)
table.to_csv(f"{RESULTS}/table_main_comparison.csv")
display(Markdown("**Main comparison on the Libimseti test set** (average of both sides, top 50; ours = mean ± std over seeds)"))
display(table)

In [ ]:
cols = ["recall@50", "ndcg@50", "crecall@50", "srecall@50", "rndcg@50", "A_recall@50", "B_recall@50"]
full = mean[cols].round(4).astype(str) + " ± " + std[cols].round(4).astype(str)
full.insert(0, "runs", n)
full.to_csv(f"{RESULTS}/table_all_metrics_ours.csv")
display(Markdown("**All metrics, our re-implementation** (mean ± std over seeds)"))
display(full)

In [ ]:
fig_comparison(mean, std, OFFICIAL, f"{RESULTS}/fig_main_comparison.png")
fig_curves({k: HIST[k] for k in ["DPGNN", "LightGCN", "BPRMF"] if k in HIST}, f"{RESULTS}/fig_training_curves.png")
fig_sides(mean.drop(index=[i for i in mean.index if i == "Popularity"]), f"{RESULTS}/fig_sides.png")
abl = [l for l in ["DPGNN", "DPGNN w/o contrastive loss", "DPGNN w/o one-sided edges", "DPGNN w/o both"] if l in mean.index]
fig_bars({l: mean.loc[l, "crecall@50"] for l in abl}, {l: std.loc[l, "crecall@50"] for l in abl},
         "Ablation: test CRecall@50 of DPGNN variants (mean ± std over seeds)", "CRecall@50",
         f"{RESULTS}/fig_ablation.png", highlight="DPGNN")
if TWOWAY:
    names = {"both": "Both directions (DPGNN score)", "man_side": "Man's interest only", "woman_side": "Woman's interest only"}
    vals = {names[m]: float(np.mean([TWOWAY[s][m]["crecall@50"] for s in TWOWAY])) for m in names}
    errs = {names[m]: float(np.std([TWOWAY[s][m]["crecall@50"] for s in TWOWAY], ddof=1)) if len(TWOWAY) > 1 else 0.0 for m in names}
    fig_bars(vals, errs, "Scoring one direction vs both: test CRecall@50 of trained DPGNN", "CRecall@50",
             f"{RESULTS}/fig_two_way_score.png", highlight=names["both"])
for f in ["fig_main_comparison", "fig_training_curves", "fig_ablation", "fig_two_way_score", "fig_sides"]:
    if os.path.exists(f"{RESULTS}/{f}.png"):
        display(Image(f"{RESULTS}/{f}.png"))

In [ ]:
# Compact summary: paste this output back to Claude to write the report
summary = {"device": str(DEVICE), "seeds": SEEDS,
           "ours": {l: {m: round(float(mean.loc[l, m]), 4) for m in cols} | {f"{m}_std": round(float(std.loc[l, m]), 4) for m in ["recall@50", "ndcg@50", "crecall@50"]}
                    for l in mean.index},
           "official": {k: {m: round(v, 4) for m, v in d.items()} for k, d in OFFICIAL.items()},
           "best_epochs": {f"{r['label']}|{r['seed']}": r["info"].get("best_epoch") for r in RUNS if r["info"]},
           "train_minutes": {f"{r['label']}|{r['seed']}": round(r["info"].get("train_seconds", 0) / 60, 1) for r in RUNS if r["info"]},
           "two_way": {s: {m: round(v["crecall@50"], 4) for m, v in d.items()} for s, d in TWOWAY.items()}}
save_json(summary, f"{RESULTS}/summary.json")
print(json.dumps(summary, indent=1))

In [ ]:
shutil.make_archive(f"{WORK}/dpgnn_results", "zip", RESULTS)
print("Saved", f"{WORK}/dpgnn_results.zip", "- download it from the Output panel")
